# Phage-Bacteria Interaction (PBI) Unified Analysis Pipeline

A comprehensive pipeline for analyzing phage-bacteria interactions by:
1. **Bacterial Surface Analysis**: Identifying bacterial surface proteins (receptors)
2. **Phage RBP Analysis**: Detecting phage receptor-binding proteins and depolymerases

## Pipeline Overview

### Part A: Bacterial Surface Protein Analysis
- SignalP: Signal peptide prediction
- CELLO: Subcellular localization
- Transmembrane: TM topology prediction
- Integration: Combined surface protein classification

### Part B: Phage RBP/DPO Analysis
- RBPdetect2: Deep learning RBP detection
- PhageDPO: Depolymerase prediction
- ESM-RBD: Receptor-binding domain clustering

---

## 1. Environment Setup

In [ ]:
# Check runtime environment
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running on Google Colab")
    # Check GPU availability
    import subprocess
    gpu_info = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    if gpu_info.returncode == 0:
        print("GPU available!")
    else:
        print("No GPU detected. Some features may be slower.")
else:
    print("Running locally")

In [ ]:
# Install dependencies
!pip install -q biopython pandas numpy scipy scikit-learn
!pip install -q matplotlib seaborn tqdm rich

# For phage pipeline (ESM-based analysis)
!pip install -q torch transformers safetensors huggingface-hub accelerate
!pip install -q fair-esm xgboost

print("\nAll dependencies installed!")

In [ ]:
# Core imports
import os
import sys
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

print("Libraries imported successfully!")

---

## 2. Utility Functions

In [ ]:
# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def setup_environment(base_dir: Optional[str] = None) -> Dict[str, Path]:
    """Setup pipeline directories."""
    if base_dir is None:
        base_dir = os.getcwd()
    
    base_path = Path(base_dir)
    
    paths = {
        'base': base_path,
        'input': base_path / 'input',
        'output': base_path / 'output',
        # Bacteria outputs
        'output_signalp': base_path / 'output' / 'bacteria' / 'signalp',
        'output_cello': base_path / 'output' / 'bacteria' / 'cello',
        'output_transmembrane': base_path / 'output' / 'bacteria' / 'transmembrane',
        'output_bacteria_integrated': base_path / 'output' / 'bacteria' / 'integrated',
        # Phage outputs
        'output_rbpdetect2': base_path / 'output' / 'phage' / 'rbpdetect2',
        'output_phagedpo': base_path / 'output' / 'phage' / 'phagedpo',
        'output_esm_rbd': base_path / 'output' / 'phage' / 'esm_rbd',
        'output_phage_integrated': base_path / 'output' / 'phage' / 'integrated',
        # Combined output
        'output_pbi': base_path / 'output' / 'pbi_analysis',
    }
    
    for name, path in paths.items():
        if name != 'base':
            path.mkdir(parents=True, exist_ok=True)
    
    return paths


def read_fasta(filepath: str) -> List[Tuple[str, str]]:
    """Read FASTA file."""
    sequences = []
    current_header = None
    current_seq = []
    
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                if current_header is not None:
                    sequences.append((current_header, ''.join(current_seq)))
                current_header = line[1:]
                current_seq = []
            elif line:
                current_seq.append(line)
        
        if current_header is not None:
            sequences.append((current_header, ''.join(current_seq)))
    
    return sequences


def write_fasta(sequences: List[Tuple[str, str]], filepath: str) -> None:
    """Write FASTA file."""
    with open(filepath, 'w') as f:
        for header, seq in sequences:
            f.write(f'>{header}\n')
            for i in range(0, len(seq), 80):
                f.write(seq[i:i+80] + '\n')


def get_device() -> str:
    """Get best available device."""
    try:
        import torch
        if torch.cuda.is_available():
            return 'cuda'
        elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
            return 'mps'
        return 'cpu'
    except ImportError:
        return 'cpu'


print(f"Device: {get_device()}")

---

## 3. Bacterial Surface Protein Analysis Modules

In [ ]:
# ============================================================================
# SIGNAL PEPTIDE PREDICTOR
# ============================================================================

class SignalPPredictor:
    """Signal peptide prediction based on N-terminal sequence features."""
    
    def __init__(self, organism: str = "gram-"):
        self.organism = organism
        self.signal_peptide_length = 30
    
    def _analyze_n_terminus(self, sequence: str) -> Dict[str, float]:
        n_term = sequence[:self.signal_peptide_length].upper()
        
        if len(n_term) < 20:
            return {'has_signal': False, 'score': 0.0, 'cleavage_site': 0}
        
        # n-region analysis
        n_region = n_term[:5]
        n_positive = sum(1 for aa in n_region if aa in 'RK')
        
        # h-region analysis
        h_region = n_term[5:20]
        h_hydrophobic_ratio = sum(1 for aa in h_region if aa in 'AILMFVWP') / len(h_region)
        
        # c-region analysis
        c_region = n_term[15:25] if len(n_term) >= 25 else n_term[15:]
        c_small = sum(1 for aa in c_region if aa in 'AGS')
        
        # Cleavage site
        cleavage_site = 0
        for i in range(15, min(30, len(n_term) - 2)):
            if n_term[i] in 'AG' and n_term[i+2] in 'AG':
                cleavage_site = i + 3
                break
        
        # Score calculation
        score = min(n_positive / 2, 0.3) + min(h_hydrophobic_ratio, 0.4)
        score += min(c_small / 5, 0.2) if c_region else 0
        score += 0.1 if cleavage_site > 0 else 0
        
        return {
            'has_signal': score > 0.5,
            'score': score,
            'cleavage_site': cleavage_site if score > 0.5 else 0,
        }
    
    def predict(self, sequences: List[Tuple[str, str]]) -> pd.DataFrame:
        print(f"[SignalP] Analyzing {len(sequences)} sequences...")
        
        results = []
        for header, seq in tqdm(sequences, desc="SignalP"):
            protein_id = header.split()[0]
            analysis = self._analyze_n_terminus(seq)
            
            sp_type = 'SP(Sec/SPI)' if analysis['has_signal'] else 'OTHER'
            
            results.append({
                'protein_id': protein_id,
                'description': header,
                'length': len(seq),
                'prediction': sp_type,
                'sp_score': analysis['score'],
                'has_signal_peptide': analysis['has_signal'],
                'cleavage_site': analysis['cleavage_site'],
            })
        
        return pd.DataFrame(results)
    
    def run(self, input_file: str, output_dir: str) -> pd.DataFrame:
        sequences = read_fasta(input_file)
        results = self.predict(sequences)
        
        output_file = Path(output_dir) / "signalp_predictions.tsv"
        results.to_csv(output_file, sep='\t', index=False)
        
        sp_count = results['has_signal_peptide'].sum()
        print(f"[SignalP] Found {sp_count}/{len(results)} proteins with signal peptides")
        
        return results

In [ ]:
# ============================================================================
# CELLO LOCALIZATION PREDICTOR
# ============================================================================

class CELLOPredictor:
    """Subcellular localization prediction."""
    
    LOCALIZATIONS = ['Cytoplasmic', 'InnerMembrane', 'Periplasmic', 'OuterMembrane', 'Extracellular']
    
    def __init__(self):
        pass
    
    def _calculate_features(self, sequence: str) -> Dict[str, float]:
        seq = sequence.upper()
        length = len(seq)
        
        if length == 0:
            return {}
        
        features = {}
        for aa in 'ACDEFGHIKLMNPQRSTVWY':
            features[f'aa_{aa}'] = seq.count(aa) / length
        
        hydrophobic, polar, positive, negative, aromatic = set('AILMFVWP'), set('STNQ'), set('RKH'), set('DE'), set('FWY')
        
        features['hydrophobic'] = sum(1 for aa in seq if aa in hydrophobic) / length
        features['polar'] = sum(1 for aa in seq if aa in polar) / length
        features['positive'] = sum(1 for aa in seq if aa in positive) / length
        features['negative'] = sum(1 for aa in seq if aa in negative) / length
        features['aromatic'] = sum(1 for aa in seq if aa in aromatic) / length
        features['length'] = length
        
        n_term = seq[:30] if length >= 30 else seq
        features['n_term_hydrophobic'] = sum(1 for aa in n_term if aa in hydrophobic) / len(n_term)
        features['n_term_positive'] = sum(1 for aa in n_term if aa in positive) / len(n_term)
        
        return features
    
    def _predict_localization(self, features: Dict[str, float]) -> Tuple[str, float]:
        scores = {loc: 0.0 for loc in self.LOCALIZATIONS}
        
        if features['hydrophobic'] < 0.35 and features['n_term_hydrophobic'] < 0.4:
            scores['Cytoplasmic'] += 0.4
        if features['hydrophobic'] > 0.4:
            scores['InnerMembrane'] += 0.5
        
        has_signal = features['n_term_hydrophobic'] > 0.5 and features['n_term_positive'] > 0.08
        
        if has_signal:
            scores['Cytoplasmic'] -= 0.3
            if features['aromatic'] > 0.08 and features['length'] > 200:
                scores['OuterMembrane'] += 0.4
            if features['length'] < 400 and features['hydrophobic'] < 0.4:
                scores['Periplasmic'] += 0.35
            if features['negative'] > 0.12 or features['length'] > 500:
                scores['Extracellular'] += 0.25
        
        best_loc = max(scores, key=scores.get)
        total = sum(scores.values())
        confidence = min(max(scores[best_loc] / total if total > 0 else 0.2, 0.1), 0.95)
        
        return best_loc, confidence
    
    def predict(self, sequences: List[Tuple[str, str]]) -> pd.DataFrame:
        print(f"[CELLO] Analyzing {len(sequences)} sequences...")
        
        results = []
        for header, seq in tqdm(sequences, desc="CELLO"):
            protein_id = header.split()[0]
            features = self._calculate_features(seq)
            localization, confidence = self._predict_localization(features)
            
            results.append({
                'protein_id': protein_id,
                'description': header,
                'length': len(seq),
                'localization': localization,
                'confidence': confidence,
            })
        
        return pd.DataFrame(results)
    
    def run(self, input_file: str, output_dir: str) -> pd.DataFrame:
        sequences = read_fasta(input_file)
        results = self.predict(sequences)
        
        output_file = Path(output_dir) / "cello_predictions.tsv"
        results.to_csv(output_file, sep='\t', index=False)
        
        print(f"[CELLO] Localization summary:")
        for loc in self.LOCALIZATIONS:
            count = (results['localization'] == loc).sum()
            print(f"  {loc}: {count}")
        
        return results

In [ ]:
# ============================================================================
# TRANSMEMBRANE PREDICTOR
# ============================================================================

class TransmembranePredictor:
    """Transmembrane topology prediction."""
    
    HYDROPHOBICITY = {
        'A': 1.8, 'R': -4.5, 'N': -3.5, 'D': -3.5, 'C': 2.5,
        'Q': -3.5, 'E': -3.5, 'G': -0.4, 'H': -3.2, 'I': 4.5,
        'L': 3.8, 'K': -3.9, 'M': 1.9, 'F': 2.8, 'P': -1.6,
        'S': -0.8, 'T': -0.7, 'W': -0.9, 'Y': -1.3, 'V': 4.2,
    }
    
    def __init__(self, window_size: int = 19, threshold: float = 1.6):
        self.window_size = window_size
        self.threshold = threshold
    
    def _calculate_profile(self, sequence: str) -> List[float]:
        seq = sequence.upper()
        half_window = self.window_size // 2
        profile = []
        
        for i in range(len(seq)):
            start, end = max(0, i - half_window), min(len(seq), i + half_window + 1)
            window = seq[start:end]
            hydro = sum(self.HYDROPHOBICITY.get(aa, 0) for aa in window) / len(window) if window else 0
            profile.append(hydro)
        
        return profile
    
    def _find_tm_regions(self, profile: List[float]) -> List[Tuple[int, int]]:
        regions = []
        in_region, start = False, 0
        
        for i, hydro in enumerate(profile):
            if hydro >= self.threshold:
                if not in_region:
                    in_region, start = True, i
            else:
                if in_region:
                    if i - start >= 15:
                        regions.append((start, i))
                    in_region = False
        
        if in_region and len(profile) - start >= 15:
            regions.append((start, len(profile)))
        
        return regions
    
    def predict(self, sequences: List[Tuple[str, str]]) -> pd.DataFrame:
        print(f"[TM] Analyzing {len(sequences)} sequences...")
        
        results = []
        for header, seq in tqdm(sequences, desc="Transmembrane"):
            protein_id = header.split()[0]
            profile = self._calculate_profile(seq)
            tm_regions = self._find_tm_regions(profile)
            tm_count = len(tm_regions)
            
            if tm_count == 0:
                aromatic_ratio = sum(1 for aa in seq.upper() if aa in 'FWY') / len(seq)
                classification = 'BETA' if aromatic_ratio > 0.1 and len(seq) > 200 else 'GLOBULAR'
            else:
                classification = f'TM{tm_count}' if tm_count <= 6 else 'TM7+'
            
            results.append({
                'protein_id': protein_id,
                'length': len(seq),
                'tm_count': tm_count,
                'classification': classification,
                'is_membrane_protein': tm_count > 0 or classification == 'BETA',
            })
        
        return pd.DataFrame(results)
    
    def run(self, input_file: str, output_dir: str) -> pd.DataFrame:
        sequences = read_fasta(input_file)
        results = self.predict(sequences)
        
        output_file = Path(output_dir) / "transmembrane_predictions.tsv"
        results.to_csv(output_file, sep='\t', index=False)
        
        membrane_count = results['is_membrane_protein'].sum()
        print(f"[TM] Membrane proteins: {membrane_count}/{len(results)}")
        
        return results

In [ ]:
# ============================================================================
# BACTERIAL RESULTS INTEGRATOR
# ============================================================================

class BacteriaResultsIntegrator:
    """Integrate bacterial surface protein predictions."""
    
    SURFACE_LOCALIZATIONS = ['OuterMembrane', 'Extracellular', 'Periplasmic', 'CellWall']
    
    def integrate(self, signalp_df: pd.DataFrame, cello_df: pd.DataFrame, tm_df: pd.DataFrame) -> pd.DataFrame:
        print("[Integrator] Combining bacterial predictions...")
        
        # Merge dataframes
        merged = signalp_df[['protein_id', 'length', 'has_signal_peptide', 'sp_score', 'cleavage_site']].merge(
            cello_df[['protein_id', 'localization', 'confidence']],
            on='protein_id', how='outer'
        ).merge(
            tm_df[['protein_id', 'tm_count', 'classification', 'is_membrane_protein']],
            on='protein_id', how='outer'
        )
        
        # Determine surface proteins
        merged['is_surface_protein'] = (
            merged['localization'].isin(self.SURFACE_LOCALIZATIONS) |
            merged['has_signal_peptide'].fillna(False) |
            (merged['classification'] == 'BETA')
        )
        
        # Calculate confidence
        merged['surface_confidence'] = 0.0
        mask = merged['is_surface_protein']
        merged.loc[mask, 'surface_confidence'] = (
            merged.loc[mask, 'confidence'].fillna(0) * 0.5 +
            merged.loc[mask, 'sp_score'].fillna(0) * 0.3 +
            (merged.loc[mask, 'classification'] == 'BETA').astype(float) * 0.2
        )
        
        surface_count = merged['is_surface_protein'].sum()
        print(f"[Integrator] Surface proteins: {surface_count}/{len(merged)}")
        
        return merged

---

## 4. Phage RBP/DPO Analysis Modules

In [ ]:
# ============================================================================
# RBP DETECTOR (Simplified version without full ESM model)
# ============================================================================

class RBPDetector:
    """Receptor-Binding Protein detection based on sequence features."""
    
    # RBP-associated keywords for annotation-based detection
    RBP_KEYWORDS = [
        'tail fiber', 'tailspike', 'receptor binding', 'rbp', 'spike',
        'baseplate', 'host specificity', 'adhesin', 'depolymerase',
        'tail protein', 'fiber protein', 'attachment'
    ]
    
    def __init__(self, threshold: float = 0.5):
        self.threshold = threshold
    
    def _calculate_rbp_features(self, sequence: str, header: str) -> Dict[str, float]:
        """Calculate RBP-associated features."""
        seq = sequence.upper()
        length = len(seq)
        
        features = {}
        
        # Length feature (RBPs tend to be longer)
        features['length_score'] = min(length / 1000, 1.0) * 0.2
        
        # Aromatic content (often high in carbohydrate-binding domains)
        aromatic = sum(1 for aa in seq if aa in 'FWY') / length
        features['aromatic_score'] = min(aromatic * 5, 0.3)
        
        # Beta-strand propensity (RBPs often have beta-structures)
        beta_formers = sum(1 for aa in seq if aa in 'VIY') / length
        features['beta_score'] = min(beta_formers * 3, 0.2)
        
        # Keyword-based scoring
        header_lower = header.lower()
        keyword_match = any(kw in header_lower for kw in self.RBP_KEYWORDS)
        features['keyword_score'] = 0.5 if keyword_match else 0.0
        
        # Total score
        features['total_score'] = sum(features.values())
        
        return features
    
    def predict(self, sequences: List[Tuple[str, str]]) -> pd.DataFrame:
        print(f"[RBP] Analyzing {len(sequences)} sequences...")
        
        results = []
        for header, seq in tqdm(sequences, desc="RBP Detection"):
            protein_id = header.split()[0]
            features = self._calculate_rbp_features(seq, header)
            
            rbp_score = min(features['total_score'], 1.0)
            is_rbp = rbp_score >= self.threshold
            
            results.append({
                'protein_id': protein_id,
                'description': header,
                'length': len(seq),
                'rbp_score': rbp_score,
                'is_rbp': is_rbp,
            })
        
        return pd.DataFrame(results)
    
    def run(self, input_file: str, output_dir: str) -> pd.DataFrame:
        sequences = read_fasta(input_file)
        results = self.predict(sequences)
        
        output_file = Path(output_dir) / "rbp_predictions.tsv"
        results.to_csv(output_file, sep='\t', index=False)
        
        rbp_count = results['is_rbp'].sum()
        print(f"[RBP] Predicted RBPs: {rbp_count}/{len(results)}")
        
        return results

In [ ]:
# ============================================================================
# DEPOLYMERASE PREDICTOR
# ============================================================================

class DepolymerasePredictor:
    """Depolymerase prediction based on sequence features."""
    
    DPO_KEYWORDS = [
        'depolymerase', 'polysaccharide', 'lyase', 'hydrolase',
        'glycosidase', 'capsule', 'pectin', 'hyaluronate',
        'tailspike', 'endosialidase'
    ]
    
    def __init__(self, threshold: float = 0.5):
        self.threshold = threshold
    
    def _calculate_dpo_features(self, sequence: str, header: str) -> Dict[str, float]:
        seq = sequence.upper()
        length = len(seq)
        
        features = {}
        
        # Length (depolymerases are typically 400-1000 aa)
        if 400 <= length <= 1000:
            features['length_score'] = 0.2
        elif 300 <= length <= 1200:
            features['length_score'] = 0.1
        else:
            features['length_score'] = 0.0
        
        # Catalytic residue enrichment (D, E for glycosidases)
        acidic = sum(1 for aa in seq if aa in 'DE') / length
        features['acidic_score'] = min(acidic * 3, 0.2)
        
        # Aromatic residues (substrate binding)
        aromatic = sum(1 for aa in seq if aa in 'FWY') / length
        features['aromatic_score'] = min(aromatic * 4, 0.2)
        
        # Keyword matching
        header_lower = header.lower()
        keyword_match = any(kw in header_lower for kw in self.DPO_KEYWORDS)
        features['keyword_score'] = 0.5 if keyword_match else 0.0
        
        features['total_score'] = sum(features.values())
        
        return features
    
    def predict(self, sequences: List[Tuple[str, str]]) -> pd.DataFrame:
        print(f"[DPO] Analyzing {len(sequences)} sequences...")
        
        results = []
        for header, seq in tqdm(sequences, desc="Depolymerase"):
            protein_id = header.split()[0]
            features = self._calculate_dpo_features(seq, header)
            
            dpo_score = min(features['total_score'], 1.0)
            is_dpo = dpo_score >= self.threshold
            
            results.append({
                'protein_id': protein_id,
                'description': header,
                'length': len(seq),
                'dpo_score': dpo_score,
                'is_depolymerase': is_dpo,
            })
        
        return pd.DataFrame(results)
    
    def run(self, input_file: str, output_dir: str) -> pd.DataFrame:
        sequences = read_fasta(input_file)
        results = self.predict(sequences)
        
        output_file = Path(output_dir) / "depolymerase_predictions.tsv"
        results.to_csv(output_file, sep='\t', index=False)
        
        dpo_count = results['is_depolymerase'].sum()
        print(f"[DPO] Predicted depolymerases: {dpo_count}/{len(results)}")
        
        return results

In [ ]:
# ============================================================================
# PHAGE RESULTS INTEGRATOR
# ============================================================================

class PhageResultsIntegrator:
    """Integrate phage RBP/DPO predictions."""
    
    def integrate(self, rbp_df: pd.DataFrame, dpo_df: pd.DataFrame) -> pd.DataFrame:
        print("[Integrator] Combining phage predictions...")
        
        merged = rbp_df[['protein_id', 'description', 'length', 'rbp_score', 'is_rbp']].merge(
            dpo_df[['protein_id', 'dpo_score', 'is_depolymerase']],
            on='protein_id', how='outer'
        )
        
        # Combined score
        merged['combined_score'] = (
            merged['rbp_score'].fillna(0) * 0.6 +
            merged['dpo_score'].fillna(0) * 0.4
        )
        
        # Classification
        def classify(row):
            if row['is_rbp'] and row['is_depolymerase']:
                return 'RBP+DPO'
            elif row['is_rbp']:
                return 'RBP'
            elif row['is_depolymerase']:
                return 'DPO'
            else:
                return 'Other'
        
        merged['classification'] = merged.apply(classify, axis=1)
        
        print(f"[Integrator] Classification summary:")
        for cls, count in merged['classification'].value_counts().items():
            print(f"  {cls}: {count}")
        
        return merged

---

## 5. Setup and Input Data

In [ ]:
# Setup directories
paths = setup_environment()

print("Directory structure created:")
for name, path in paths.items():
    print(f"  {name}: {path}")

In [ ]:
# Upload or set input files
if IN_COLAB:
    from google.colab import files
    print("Upload your FASTA files:")
    print("  - Bacterial proteins (.faa)")
    print("  - Phage proteins (.faa)")
    uploaded = files.upload()
    
    for filename in uploaded.keys():
        dest = paths['input'] / filename
        os.rename(filename, str(dest))
        print(f"Saved: {dest}")
else:
    print("Place your FASTA files in the 'input/' directory")

In [ ]:
# Set input file paths
# Modify these paths to match your uploaded files

BACTERIA_FASTA = str(paths['input'] / 'bacteria_proteins.faa')  # Change filename as needed
PHAGE_FASTA = str(paths['input'] / 'phage_proteins.faa')        # Change filename as needed

# Organism type for bacteria
ORGANISM = 'gram-'  # Options: 'gram-', 'gram+'

# List available input files
print("Available input files:")
for f in paths['input'].glob('*.faa'):
    print(f"  {f.name}")
for f in paths['input'].glob('*.fasta'):
    print(f"  {f.name}")

---

## 6. Run Bacterial Surface Analysis

In [ ]:
# Check if bacterial input exists
if os.path.exists(BACTERIA_FASTA):
    bacteria_sequences = read_fasta(BACTERIA_FASTA)
    print(f"Loaded {len(bacteria_sequences)} bacterial proteins")
else:
    print(f"File not found: {BACTERIA_FASTA}")
    print("Please upload your bacterial protein FASTA file")

In [ ]:
# Run SignalP
signalp = SignalPPredictor(organism=ORGANISM)
signalp_results = signalp.run(BACTERIA_FASTA, str(paths['output_signalp']))

In [ ]:
# Run CELLO
cello = CELLOPredictor()
cello_results = cello.run(BACTERIA_FASTA, str(paths['output_cello']))

In [ ]:
# Run Transmembrane
tm = TransmembranePredictor()
tm_results = tm.run(BACTERIA_FASTA, str(paths['output_transmembrane']))

In [ ]:
# Integrate bacterial results
bacteria_integrator = BacteriaResultsIntegrator()
bacteria_integrated = bacteria_integrator.integrate(signalp_results, cello_results, tm_results)

# Save
bacteria_output = paths['output_bacteria_integrated'] / 'surface_proteins.tsv'
bacteria_integrated.to_csv(bacteria_output, sep='\t', index=False)
print(f"Saved to: {bacteria_output}")

# Display
display(bacteria_integrated.head(10))

---

## 7. Run Phage RBP/DPO Analysis

In [ ]:
# Check if phage input exists
if os.path.exists(PHAGE_FASTA):
    phage_sequences = read_fasta(PHAGE_FASTA)
    print(f"Loaded {len(phage_sequences)} phage proteins")
else:
    print(f"File not found: {PHAGE_FASTA}")
    print("Please upload your phage protein FASTA file")

In [ ]:
# Run RBP detection
rbp_detector = RBPDetector(threshold=0.4)
rbp_results = rbp_detector.run(PHAGE_FASTA, str(paths['output_rbpdetect2']))

In [ ]:
# Run Depolymerase prediction
dpo_predictor = DepolymerasePredictor(threshold=0.4)
dpo_results = dpo_predictor.run(PHAGE_FASTA, str(paths['output_phagedpo']))

In [ ]:
# Integrate phage results
phage_integrator = PhageResultsIntegrator()
phage_integrated = phage_integrator.integrate(rbp_results, dpo_results)

# Save
phage_output = paths['output_phage_integrated'] / 'rbp_dpo_predictions.tsv'
phage_integrated.to_csv(phage_output, sep='\t', index=False)
print(f"Saved to: {phage_output}")

# Display
display(phage_integrated.head(10))

---

## 8. Combined PBI Analysis Summary

In [ ]:
# Generate summary
print("=" * 70)
print("  PHAGE-BACTERIA INTERACTION (PBI) ANALYSIS SUMMARY")
print("=" * 70)

print("\n[BACTERIAL SURFACE PROTEINS]")
print(f"  Total proteins analyzed: {len(bacteria_integrated)}")
surface_count = bacteria_integrated['is_surface_protein'].sum()
print(f"  Surface proteins identified: {surface_count} ({100*surface_count/len(bacteria_integrated):.1f}%)")
print("\n  Localization breakdown:")
for loc, count in bacteria_integrated['localization'].value_counts().items():
    print(f"    {loc}: {count}")

print("\n[PHAGE RBP/DPO PROTEINS]")
print(f"  Total proteins analyzed: {len(phage_integrated)}")
rbp_count = phage_integrated['is_rbp'].sum()
dpo_count = phage_integrated['is_depolymerase'].sum()
print(f"  Predicted RBPs: {rbp_count}")
print(f"  Predicted Depolymerases: {dpo_count}")
print("\n  Classification breakdown:")
for cls, count in phage_integrated['classification'].value_counts().items():
    print(f"    {cls}: {count}")

print("\n" + "=" * 70)

In [ ]:
# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Bacterial localization
loc_counts = bacteria_integrated['localization'].value_counts()
axes[0, 0].pie(loc_counts, labels=loc_counts.index, autopct='%1.1f%%')
axes[0, 0].set_title('Bacterial Protein Localization')

# Surface vs non-surface
surface_counts = bacteria_integrated['is_surface_protein'].value_counts()
axes[0, 1].bar(['Non-surface', 'Surface'], [surface_counts.get(False, 0), surface_counts.get(True, 0)])
axes[0, 1].set_title('Surface Protein Distribution')
axes[0, 1].set_ylabel('Count')

# Phage classification
cls_counts = phage_integrated['classification'].value_counts()
axes[1, 0].bar(cls_counts.index, cls_counts.values, color=['steelblue', 'coral', 'green', 'gray'][:len(cls_counts)])
axes[1, 0].set_title('Phage Protein Classification')
axes[1, 0].set_ylabel('Count')

# Score distributions
axes[1, 1].hist(phage_integrated['rbp_score'], bins=20, alpha=0.6, label='RBP Score')
axes[1, 1].hist(phage_integrated['dpo_score'], bins=20, alpha=0.6, label='DPO Score')
axes[1, 1].axvline(x=0.4, color='r', linestyle='--', label='Threshold')
axes[1, 1].set_xlabel('Score')
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_title('Phage Prediction Score Distribution')
axes[1, 1].legend()

plt.tight_layout()
plt.savefig(paths['output_pbi'] / 'pbi_summary.png', dpi=150)
plt.show()

---

## 9. Export Results

In [ ]:
# High-confidence candidates
high_conf_bacteria = bacteria_integrated[
    (bacteria_integrated['is_surface_protein']) & 
    (bacteria_integrated['surface_confidence'] > 0.3)
].sort_values('surface_confidence', ascending=False)

high_conf_phage = phage_integrated[
    (phage_integrated['classification'] != 'Other') &
    (phage_integrated['combined_score'] > 0.4)
].sort_values('combined_score', ascending=False)

print(f"High-confidence bacterial surface proteins: {len(high_conf_bacteria)}")
print(f"High-confidence phage RBP/DPO proteins: {len(high_conf_phage)}")

# Save high-confidence results
high_conf_bacteria.to_csv(paths['output_pbi'] / 'high_conf_surface_proteins.tsv', sep='\t', index=False)
high_conf_phage.to_csv(paths['output_pbi'] / 'high_conf_rbp_dpo.tsv', sep='\t', index=False)

In [ ]:
if IN_COLAB:
    from google.colab import files
    
    # Create zip of all results
    !zip -r pbi_results.zip output/
    
    files.download('pbi_results.zip')
    print("Results downloaded!")
else:
    print(f"Results saved in: {paths['output']}")
    print("\nOutput files:")
    for f in paths['output'].rglob('*.tsv'):
        print(f"  {f.relative_to(paths['output'])}")

---

## License

MIT License

---

## Citation

If you use this pipeline, please cite:
- [Your paper/repository here]